# P11 · Calendario de medición de un indicador oficial

> **¿Qué días de 2026 entran en la medición según las reglas oficiales?**

Metodología DIPRES, listado versionado de feriados y calendario gregoriano.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/calendario_medicion/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py 


## 1 · El problema

El denominador de un indicador oficial no es «los días del año». La metodología
descarta fines de semana, feriados, tres meses completos, vacaciones de invierno, la
semana de fiestas patrias y otras fechas. Además, la medición ocurre solo en cuatro
meses. Una nota al pie decide qué observaciones pueden entrar al indicador.

El riesgo no está en restar días, sino en sumar exclusiones solapadas. En 2026 las ocho
reglas marcan 256 días, pero el calendario contiene 365 y solo 201 quedan excluidos: 55
marcas se superponen. El cuaderno construye cada regla como una máscara auditable y
mide su aporte al denominador final.


## 2 · La solución que se va a aplicar

Se valida primero el listado de feriados contra Pascua y fechas fijas. Luego se genera
un calendario diario, se aplican las ocho reglas y se auditan sus solapamientos. El
resultado se cruza con los meses efectivos de medición. Por último se compara 2024–2027
para distinguir redundancias estructurales de coincidencias accidentales y se somete a
sensibilidad el único supuesto no fijado por la fuente: las vacaciones de invierno.


## Implementaciones comparables

La ruta principal usa Pandas. Polars y R recalculan el indicador central desde el contrato de salida versionado y generan el mismo gráfico.


### Polars + Matplotlib

```python
"""Contraste Polars y visualización Matplotlib del caso calendario_medicion."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "calendario_medicion" / "resultados" / "comparacion_anios.csv"


def analizar() -> pl.DataFrame:
    datos = pl.read_csv(SOURCE)
    return datos.sort("anio").select(["anio", "dias_medibles"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    x = resultado["anio"].to_list()
    y = resultado["dias_medibles"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Año")
    ax.set_ylabel("Días medibles (recuento)")
    ax.set_ylim(bottom=0)
    ax.set_title("El denominador cambia con el calendario anual")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P11 · Período: 2024–2027",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--grafico", type=Path)
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/calendario_medicion/cuaderno/solucion_polars.py`.


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Contraste R + ggplot2 del caso calendario_medicion.
script_arg <- grep("^--file=", commandArgs(trailingOnly = FALSE), value = TRUE)
script_path <- if (length(script_arg)) sub("^--file=", "", script_arg[1]) else "cuaderno/solucion.R"
root <- normalizePath(file.path(dirname(script_path), "../../.."), mustWork = TRUE)
source_csv <- file.path(root, "cases", "calendario_medicion", "resultados", "comparacion_anios.csv")

datos <- read.csv(source_csv, check.names = FALSE, fileEncoding = "UTF-8")
resultado <- datos[order(datos$anio), c("anio", "dias_medibles")]
print(resultado, row.names = FALSE)

args <- commandArgs(trailingOnly = TRUE)
plot_pos <- match("--grafico", args)
if (!is.na(plot_pos)) {
  if (plot_pos == length(args)) stop("--grafico necesita una ruta de salida")
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  destino <- args[plot_pos + 1]
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["anio"]], y = .data[["dias_medibles"]])
  ) +
    ggplot2::geom_line(ggplot2::aes(group = 1), linewidth = 1, colour = "#11253B") +
    ggplot2::geom_point(size = 2.5, colour = "#55D6BE") +
    ggplot2::labs(
      title = "El denominador cambia con el calendario anual",
      x = "Año", y = "Días medibles (recuento)",
      caption = "Fuente: Resultados reproducibles del caso P11 · Período: 2024–2027"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(destino, grafico, width = 9, height = 5, dpi = 150)
  cat(destino, "\n")
}
```

Ejecutar con `Rscript cases/calendario_medicion/cuaderno/solucion.R`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.calendario_medicion import solution as p11

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Validar la fuente de feriados


El insumo externo se confronta con Pascua calculada y diez fechas fijas.

> ⚠️ Un CSV descargado no se vuelve verdadero por estar bien formado.


In [ ]:
resultados = p11.analizar(2026)
resultados['validacion_feriados']


### Paso 2 · Declarar las ocho reglas antes de calcular


Cada exclusión conserva nombre, descripción y fuente.

> ⚠️ Sumar los excluidos por regla cuenta 55 días más de una vez.


In [ ]:
resultados['exclusion_por_regla'][['regla','excluye','exclusiva','solapada','aporte_pct']]


### Paso 3 · Construir el calendario diario


Una fila por fecha permite auditar por qué cada día entra o sale.

**En Excel:** `=SECUENCIA(365;1;FECHA(2026;1;1);1)`

Debe dar: **365 fechas**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Trabajar solo con totales impide explicar un día particular.


In [ ]:
resultados['resumen_calendario']


### Paso 4 · Medir solapamientos y redundancias


Una regla puede excluir días y aun no aportar ninguno dado el resto.

> ⚠️ Que una regla sea redundante en un año no demuestra que sea innecesaria siempre.


In [ ]:
resultados['reglas_redundantes']


### Paso 5 · Cruzar con los meses efectivos de medición


El denominador real es la intersección, no todos los días elegibles del año.

**En Excel:** `=Y(C2;ESNUMERO(COINCIDIR(MES(A2);{3;6;9;11};0)))`

Debe dar: **78 días medibles**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Publicar 164 días elegibles responde otra pregunta; solo 78 están en meses de medición.


In [ ]:
resultados['resumen_medicion']


### Paso 6 · Cuantificar el aporte al denominador final


Se quita una regla a la vez y se recalcula la intersección.

> ⚠️ Meses completos excluye 90 días del año y cero días del denominador medido.


In [ ]:
resultados['aporte_sobre_medibles']


### Paso 7 · Distinguir redundancia estructural de accidental


Cuatro años muestran qué reglas jamás afectan y cuáles dependen del calendario.

> ⚠️ Feriados no afecta en 2026, pero sí elimina entre uno y tres días en otros años.


In [ ]:
resultados['clasificacion_reglas']


### Paso 8 · Medir el supuesto de vacaciones de invierno


La fecha no publicada se desplaza ±14 días para comprobar si cambia la decisión.

> ⚠️ Mover el supuesto a junio reduce el denominador hasta seis días; no es un detalle narrativo.


In [ ]:
resultados['sensibilidad_invierno']


## 4 · El resultado

De 261 días laborales quedan 164 elegibles y 78 medibles. Cuatro reglas no cambian el denominador de 2026, pero solo tres son estructuralmente redundantes en 2024–2027.

## 5 · Qué queda aprendido

1. El denominador es una decisión documentada y debe poder explicarse fecha por fecha.
2. Las exclusiones se combinan como conjuntos; sumar conteos ignora sus intersecciones.
3. Una redundancia de un año puede ser accidental.
4. Todo supuesto de calendario se somete a sensibilidad antes de publicar.
